In [3]:
import numpy as np
X = np.random.randn(1000000)
bit = 3
x_in_range = np.sum(X >= -2**(bit-1) and X <= 2**(bit-1) - 1)
print(x_in_range)

In [93]:
def make_share(x, bitlength):
    r = np.random.randint(0, 2**(bitlength)-1, size=x.shape)
    return r.astype(np.int64), ((x - r) % (2**(bitlength))).astype(np.int64)

def reconstruct(x1, x2, bitlength):
    return signed_mod(x1 + x2, bitlength).astype(np.int64)

def signed_mod(x, bitlength):
    return (x + 2**(bitlength-1)) % (2**(bitlength)) - 2**(bitlength-1)

def truncate_and_reduce(x1,x2, bitlength, s):
    x = reconstruct(x1, x2, bitlength)
    x = x // (2**s)  # Using integer division for right shift   
    return make_share(x, bitlength-s)

def truncate_and_reduce_round(x1,x2, bitlength, s, gt1 = None, gt2=None):
    x = reconstruct(x1, x2, bitlength)
    if gt1 is not None:
        print("error:",np.max(np.abs(x-gt1)))
    x = np.round(x / (2**s))  # Using integer division for right shift   
    if gt2 is not None:
        print("error:",np.max(np.abs(x-gt2)))
    return make_share(x, bitlength-s)

In [ ]:
### simulate attention
# plaintext
b_mpc = 60 ### first need to extend to 60 bits
b_acc = 20
b_x = 4
scale_mpc = 2 ** (b_mpc - b_acc - 1)
# Generate normal distribution with mean 0 and std = 2**(b_acc-2)
std = 2**(b_acc-2)  # Using quarter of range as std to ensure most values fall within bounds
tmp1 = np.random.normal(0, std, size=(128,4096))
tmp2 = np.random.normal(0, std, size=(128,4096))

# Clip values to maintain the same bounds as before
tmp1 = np.clip(tmp1, -2**(b_acc-1), 2**(b_acc-1)-1)
tmp2 = np.clip(tmp2, -2**(b_acc-1), 2**(b_acc-1)-1)

# Round to integers since we're working with discrete values
tmp1 = np.round(tmp1).astype(np.int64)  ### tmp1 = Q_f/s_qf
tmp2 = np.round(tmp2).astype(np.int64)  ### tmp2 = Q_f/s_kf

s_qf = np.random.uniform(0, 1e-5, size=(1, tmp1.shape[1])) ### pre scale for Q_f
s_kf = np.random.uniform(0, 1e-5, size=(1, tmp2.shape[1])) ### pre scale for K_f
Q_f = tmp1 * s_qf
K_f = tmp2 * s_kf
# print(Q_f)
QK = Q_f @ K_f.T
### generate fake scale
s_q = np.max(np.abs(Q_f)) / 2 ** (b_x-1)
s_k = np.max(np.abs(K_f)) / 2 ** (b_x-1)
beta = 2 ** (b_x-1)
# print("np.max(s_qf / s_q): ", np.max(s_qf / s_q))
tmp11, tmp12 = make_share(tmp1, b_mpc)
print(np.max(np.abs(signed_mod(tmp11+tmp12, b_mpc) - tmp1))) ### should be 0

tmp21, tmp22 = make_share(tmp2, b_mpc)
print(np.max(np.abs(signed_mod(tmp21+tmp22, b_mpc) - tmp2))) ### should be 0

scaled_s_q = np.round(s_qf / s_q * scale_mpc).astype(np.int64)
scaled_s_k = np.round(s_kf / s_k * scale_mpc).astype(np.int64)

scaled_beta = np.round(beta * scale_mpc).astype(np.int64)
Q_q = np.round(tmp1 * s_qf / s_q) + beta
K_q = np.round(tmp2 * s_kf / s_k) + beta
Q_q_fixed = np.round(tmp1 * scaled_s_q / scale_mpc) + beta
print("error rate from fixed point: ", np.sum(np.abs(Q_q - Q_q_fixed)) / Q_q.size)   ### should near 0

K_q_fixed = np.round(tmp2 * scaled_s_k / scale_mpc) + beta
print("error rate from fixed point: ", np.sum(np.abs(K_q - K_q_fixed)) / K_q.size)   ### should near 0

### requant in MPC with local truncation to recover integer
tmp11 = tmp11 * scaled_s_q
tmp12 = tmp12 * scaled_s_q 

tmp21 = tmp21 * scaled_s_k
tmp22 = tmp22 * scaled_s_k 

tmp11, tmp12 = truncate_and_reduce_round(tmp11, tmp12, b_mpc, b_mpc - b_acc - 1)
tmp21, tmp22 = truncate_and_reduce_round(tmp21, tmp22, b_mpc, b_mpc - b_acc - 1)
tmp11 = tmp11 + beta
tmp21 = tmp21 + beta

tmp11 = signed_mod(tmp11, b_acc)
tmp12 = signed_mod(tmp12, b_acc)
tmp21 = signed_mod(tmp21, b_acc)
tmp22 = signed_mod(tmp22, b_acc)

# print(tmp11)
### find the exact difference between Q_q and Q_f
print(np.max(np.abs(reconstruct(tmp11, tmp12, b_acc) - Q_q)))  ### should be 0
print(np.max(np.abs(reconstruct(tmp21, tmp22, b_acc) - K_q)))  ### should be 0
idx = np.where(np.abs(reconstruct(tmp11, tmp12, b_acc) - Q_q) > 0.01)
ans = reconstruct(tmp11, tmp12, b_acc)
print("error rate of Q_q, K_q: ", np.sum(np.abs(ans-Q_q)) / ans.size)  ### should be 0


QK_q = (Q_q - beta) @ (K_q - beta).T
QK_q_fixed = QK_q * s_q * s_k
print(np.max(np.abs(QK_q_fixed - QK)))  ### this is quantization error, do not need to care, just regard QK_q_fixed as groud truth
s = QK_q_fixed - np.max(QK_q_fixed,axis=1,keepdims=True)
s = np.exp(s)/np.sum(np.exp(s),axis=1,keepdims=True)



0
0
[[-0.26317386  1.13813     0.50686975 ... -1.77721136  0.50209652
   0.21425796]
 [ 0.68277249 -2.02328734 -0.16942868 ...  0.87047446 -0.82334612
  -0.30347252]
 [-0.55440842  0.13790545  1.15797612 ...  2.96623177  0.69555739
  -0.11600362]
 ...
 [ 0.47535781  0.52010442 -1.61715121 ... -4.77050056  0.16273005
  -0.99893627]
 [-0.81275496  1.67868772 -1.12624497 ... -3.77365049 -0.6594114
  -0.39638445]
 [-0.08227579  0.7868464   0.38591168 ...  2.18722042 -0.34031967
   1.6812492 ]]
[[ 0.          1.31046582  0.65523291 ... -1.96569873  0.65523291
   0.        ]
 [ 0.65523291 -1.96569873  0.         ...  0.65523291 -0.65523291
   0.        ]
 [-0.65523291  0.          1.31046582 ...  3.27616454  0.65523291
   0.        ]
 ...
 [ 0.65523291  0.65523291 -1.31046582 ... -4.58663036  0.
  -1.31046582]
 [-0.65523291  1.96569873 -1.31046582 ... -3.93139745 -0.65523291
  -0.65523291]
 [ 0.          0.65523291  0.65523291 ...  1.96569873 -0.65523291
   1.96569873]]
error rate from fixed